# 🌍 Avant / Après : Sites d'extraction et de remblais

Ce notebook utilise **Google Earth Engine** via **geemap** pour visualiser l'évolution de sites d'extraction (carrières, mines, cimenteries) et de remblais (terres gagnées sur l'eau) entre une période **"avant" (2018)** et une période **"après" (2026)**.

Les images proviennent de **Sentinel-2** (MSI, niveau L2A, réflectance de surface — collection harmonisée `COPERNICUS/S2_SR_HARMONIZED`), à une résolution native de **10 m**. Le contrôle **split-panel** permet de faire glisser un curseur pour comparer les deux époques côte à côte.

Si l'année cible (2018 ou 2026) ne présente pas d'images de bonne qualité (trop de nuages ou absence de couverture) sur un site donné, le notebook recherche automatiquement l'année la plus proche disponible **autour** de cette cible (± plusieurs années, en priorisant l'année la plus proche).

---

## Sites couverts

### 🪨 Sites d'extraction
| # | Site | Localisation | Type | Avant | Après |
|---|------|-------------|------|-------|-------|
| 1 | Carrare | Italie | Carrière de marbre | 2018* | 2026* |
| 2 | Salève | France/Suisse | Carrière & sablière | 2018* | 2026* |
| 3 | La Araña – Cementos Cosmos | Espagne | Carrière & cimenterie | 2018* | 2026* |
| 4 | Mont-Saint-Hilaire | Québec, Canada | Carrière de granulats | 2018* | 2026* |
| 5 | Siam Cement | Lampang, Thaïlande | Cimenterie | 2018* | 2026* |
| 6 | Île Obi (Kawasi) | Indonésie | Carrières de nickel | 2018* | 2026* |
| 7 | Montagne calcaire | Vietnam | Carrières de granite | 2018* | 2026* |
| 8 | Makrana | Rajasthan, Inde | Carrières de marbre | 2018* | 2026* |
| 9 | Ruashi | RDC | Mines de cuivre & cobalt | 2018* | 2026* |

### 🏝️ Sites de remblais
| # | Site | Localisation | Avant | Après |
|---|------|-------------|-------|-------|
| 10 | Maldives – Malé | Océan Indien | 2018* | 2026* |
| 11 | Genève – Baby-Plage | Suisse | 2018* | 2026* |
| 12 | Singapour | Asie du SE | 2018* | 2026* |
| 13 | Lagos – Eko Atlantic City | Nigeria | 2018* | 2026* |
| 14 | Cát Bà – Cái Giá | Vietnam | 2018* | 2026* |
| 15 | Hong Kong – 3ᵉ piste (aéroport) | Chine | 2018* | 2026* |
| 16 | Manille – Pasay Harbor City | Philippines | 2018* | 2026* |
| 17 | Abidjan – lagune | Côte d'Ivoire | 2018* | 2026* |

\* année effective recherchée à partir de la cible ; peut légèrement varier si l'année cible manque d'images exploitables (voir § 3.1).

## 1. Installation et imports

In [ ]:
# Décommenter si nécessaire :
# !pip install geemap earthengine-api -q

import ee
import geemap
import os
import math
from pathlib import Path

print(f"geemap version : {geemap.__version__}")

## 2. Authentification Google Earth Engine

In [ ]:
# Première utilisation : ee.Authenticate() ouvre une page d'autorisation dans le navigateur
# ee.Authenticate()
ee.Initialize(project='geemap-496017')  # ← Remplacez par votre projet GEE

## 3. Fonctions utilitaires

### 3.1 Collection Sentinel-2 et recherche d'année

- **Satellite unique** : Sentinel-2 MSI, niveau 2A (réflectance de surface), collection harmonisée `COPERNICUS/S2_SR_HARMONIZED` (couverture globale fiable à partir de 2017).
- **Résolution native** : 10 m (bandes B4/B3/B2 utilisées ici).
- **Masque nuages** : bande `SCL` (Scene Classification Layer) — masque les ombres de nuages et les nuages (probabilité moyenne/haute + cirrus fins).
- **Filtrage métadonnées** : `CLOUDY_PIXEL_PERCENTAGE` < 20 % (fallback 40 % sur l'année complète si besoin).
- **Année "avant"** : cible **2018**. Si peu/pas d'images exploitables, recherche l'année la plus proche (± 4 ans, ordre de proximité : 2018, 2017, 2019, 2016, 2020…).
- **Année "après"** : cible **2026**. Même logique de recherche autour de la cible.

La ROI est un **rectangle** défini par une demi-largeur et une demi-hauteur en mètres, centré sur le site. Cela permet d'adapter précisément la zone affichée à l'échelle de la carte sans introduire un artefact circulaire, et de caler l'emprise sur le ratio du format d'export A1 (voir § 6).

In [ ]:
# ─── Collection Sentinel-2 ────────────────────────────────────────────────────

S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'
S2_RGB_BANDS  = ['B4', 'B3', 'B2']   # Rouge, Vert, Bleu
S2_NAME       = 'Sentinel-2'

# Plage d'années couverte de façon fiable par la collection harmonisée dans GEE
S2_FIRST_YEAR = 2017
S2_LAST_YEAR  = 2026   # année en cours


def mask_s2_clouds(image):
    """Masque les nuages, ombres et cirrus à partir de la bande SCL (Scene Classification Layer)."""
    scl = image.select('SCL')
    # 3 = ombre de nuage, 8 = nuage (proba. moyenne), 9 = nuage (proba. haute), 10 = cirrus fin
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
    )
    return image.updateMask(mask)


def apply_scale_factor_s2(image):
    """Facteur d'échelle Sentinel-2 SR : DN → réflectance (0–1)."""
    return image.addBands(
        image.select('B.*').multiply(0.0001),
        overwrite=True
    )


# ─── ROI rectangulaire ────────────────────────────────────────────────────────

def make_rect_roi(lon, lat, half_w_m, half_h_m):
    """
    Crée un rectangle ee.Geometry centré sur (lon, lat).

    Paramètres
    ----------
    lon, lat   : centre en degrés décimaux
    half_w_m   : demi-largeur en mètres (axe Est-Ouest)
    half_h_m   : demi-hauteur en mètres (axe Nord-Sud)
    """
    lat_rad   = math.radians(lat)
    delta_lon = (half_w_m / (111_320 * math.cos(lat_rad)))
    delta_lat = (half_h_m / 110_540)
    return ee.Geometry.Rectangle([
        lon - delta_lon, lat - delta_lat,
        lon + delta_lon, lat + delta_lat
    ])


# ─── Recherche de l'année exploitable la plus proche d'une cible ──────────────

def _find_year_with_images(roi, candidate_years, max_cloud=20):
    """
    Parcourt candidate_years dans l'ordre (déjà trié par proximité à la cible)
    et retourne le premier qui possède au moins une image avec
    CLOUDY_PIXEL_PERCENTAGE < max_cloud sur la fenêtre avril–octobre.
    Tente ensuite toute l'année avec CC < 40 % comme fallback.
    """
    for yr in candidate_years:
        col = (
            ee.ImageCollection(S2_COLLECTION)
            .filterBounds(roi)
            .filterDate(f'{yr}-04-01', f'{yr}-10-31')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
        )
        if col.size().getInfo() > 0:
            return yr
    # Fallback : élargir à l'année complète et au seuil de nuages
    for yr in candidate_years:
        col = (
            ee.ImageCollection(S2_COLLECTION)
            .filterBounds(roi)
            .filterDate(f'{yr}-01-01', f'{yr}-12-31')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
        )
        if col.size().getInfo() > 0:
            print(f"  ⚠️  Année {yr} : filtre élargi (année complète, CC < 40 %)")
            return yr
    return candidate_years[0]  # dernier recours


def find_year_around(roi, target_year, half_window=4, max_cloud=20):
    """
    Retourne l'année la plus proche de target_year (dans un rayon de
    ± half_window ans, borné à [S2_FIRST_YEAR, S2_LAST_YEAR]) possédant
    des images Sentinel-2 exploitables.

    Les candidats sont ordonnés par proximité croissante à la cible :
    target_year, target_year-1, target_year+1, target_year-2, target_year+2, …
    """
    candidates = [target_year]
    for delta in range(1, half_window + 1):
        earlier, later = target_year - delta, target_year + delta
        if earlier >= S2_FIRST_YEAR:
            candidates.append(earlier)
        if later <= S2_LAST_YEAR:
            candidates.append(later)
    return _find_year_with_images(roi, candidates, max_cloud)


def get_sentinel2_mosaic(lon, lat, target_year, half_w_m, half_h_m):
    """
    Crée une mosaïque médiane Sentinel-2 (RGB, réflectance) pour un site,
    une emprise et une année cible donnés. Si l'année cible manque
    d'images exploitables, recherche automatiquement l'année la plus
    proche disponible (voir find_year_around).

    Retourne (image, bandes_RGB, satellite, année_effective, roi)
    """
    roi  = make_rect_roi(lon, lat, half_w_m, half_h_m)
    year = find_year_around(roi, target_year)

    start = f'{year}-04-01'
    end   = f'{year}-10-31'

    col = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
        .map(apply_scale_factor_s2)
        .select(S2_RGB_BANDS)
    )

    # Fallback : élargir à toute l'année si pas assez d'images
    if col.size().getInfo() == 0:
        col = (
            ee.ImageCollection(S2_COLLECTION)
            .filterBounds(roi)
            .filterDate(f'{year}-01-01', f'{year}-12-31')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
            .map(mask_s2_clouds)
            .map(apply_scale_factor_s2)
            .select(S2_RGB_BANDS)
        )
        print(f"  ⚠️  Peu d'images : filtre élargi à l'année complète (CC < 40 %)")

    mosaic = col.median().clip(roi)
    return mosaic, S2_RGB_BANDS, S2_NAME, year, roi


# Paramètres de visualisation communs (couleurs naturelles)
VIS = {'min': 0.0, 'max': 0.3, 'gamma': 1.4}

### 3.2 Carte split-panel

In [ ]:
def make_split_map(site_name, lon, lat, year_before=2018, year_after=2026,
                   half_w_m=4000, half_h_m=3000, zoom=13):
    """
    Génère une carte geemap avec curseur split avant/après (Sentinel-2).

    Paramètres
    ----------
    site_name   : nom du site (affiché dans les labels)
    lon, lat    : centre du site en degrés décimaux
    year_before : année cible "avant" (par défaut 2018)
    year_after  : année cible "après" (par défaut 2026)
    half_w_m    : demi-largeur de l'emprise rectangulaire en mètres
    half_h_m    : demi-hauteur de l'emprise rectangulaire en mètres
    zoom        : niveau de zoom initial de la carte

    Retourne
    --------
    m           : objet geemap.Map prêt à afficher
    meta        : dict avec les métadonnées (années, satellites, roi)
    """
    print(f"\n🔍  {site_name}")

    print(f"    Avant (cible {year_before})…", end=' ', flush=True)
    img_b, bands_b, sat_b, yr_b, roi = get_sentinel2_mosaic(
        lon, lat, year_before, half_w_m, half_h_m
    )
    print(f"OK → {yr_b} ({sat_b})")

    print(f"    Après (cible {year_after})…", end=' ', flush=True)
    img_a, bands_a, sat_a, yr_a, _   = get_sentinel2_mosaic(
        lon, lat, year_after, half_w_m, half_h_m
    )
    print(f"OK → {yr_a} ({sat_a})")

    left_layer  = geemap.ee_tile_layer(img_b, VIS, f"{yr_b} – {sat_b}")
    right_layer = geemap.ee_tile_layer(img_a, VIS, f"{yr_a} – {sat_a}")

    m = geemap.Map(center=[lat, lon], zoom=zoom, height='560px')
    m.split_map(
        left_layer,
        right_layer,
        left_label=f"◀ {yr_b}",
        right_label=f"{yr_a} ▶"
    )

    meta = dict(
        site_name=site_name, lon=lon, lat=lat,
        yr_b=yr_b, yr_a=yr_a,
        sat_b=sat_b, sat_a=sat_a,
        img_b=img_b, img_a=img_a,
        roi=roi, half_w_m=half_w_m, half_h_m=half_h_m
    )
    return m, meta

---
## 4. 🪨 Sites d'extraction

### 4.1 Carrare, Italie – Carrière de marbre

> Les Alpes Apuanes abritent les carrières de marbre blanc les plus célèbres au monde. Entre 2018 et 2026, l'extraction intensive continue d'élargir les fronts de taille, visibles comme des taches claires caractéristiques sur Sentinel-2.

In [ ]:
m_carrare, meta_carrare = make_split_map(
    site_name   = "Carrare – Carrière de marbre (Italie)",
    lon         = 10.1368,   # 10°08'12.5"E
    lat         = 44.0981,   # 44°05'53.1"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 5000,      # emprise : ~10 km × 8 km
    half_h_m    = 3530,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_carrare

### 4.2 Salève, France/Suisse – Carrière & sablière

> Le mont Salève, aux portes de Genève, abrite plusieurs carrières sur ses flancs. La résolution Sentinel-2 (10 m) permet de bien distinguer les zones dénudées.

In [ ]:
m_saleve, meta_saleve = make_split_map(
    site_name   = "Salève – Carrière & sablière (France/Suisse)",
    lon         = 6.1898,    # 6°11'23.4"E
    lat         = 46.1591,   # 46°09'32.6"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 3500,
    half_h_m    = 2470,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_saleve

### 4.3 La Araña – Cementos Cosmos, Espagne

> Cimenterie et carrière calcaire à l'est de Málaga. L'exploitation s'est étendue vers les falaises côtières.

In [ ]:
m_arana, meta_arana = make_split_map(
    site_name   = "La Araña – Cementos Cosmos (Málaga, Espagne)",
    lon         = -4.3255,   # 4°19'31.6"W
    lat         = 36.7189,   # 36°43'08.0"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 3500,
    half_h_m    = 2470,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_arana

### 4.4 Mont-Saint-Hilaire, Québec, Canada – Carrière de granulats

In [ ]:
m_msh, meta_msh = make_split_map(
    site_name   = "Mont-Saint-Hilaire – Carrière de granulats (Québec)",
    lon         = -73.1833,
    lat         = 45.5500,
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 3000,
    half_h_m    = 2120,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_msh

### 4.5 Siam Cement, province de Lampang, Thaïlande

In [ ]:
m_siam, meta_siam = make_split_map(
    site_name   = "Siam Cement – Lampang (Thaïlande)",
    lon         = 99.5868,   # 99°35'12.3"E
    lat         = 18.5491,   # 18°32'56.7"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 5000,
    half_h_m    = 3530,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_siam

### 4.6 Île Obi (Kawasi), Indonésie – Carrières de nickel

> Kawasi, sur l'île Obi (Moluques), est devenu un pôle majeur d'extraction de nickel — matière première pour l'acier inoxydable et les batteries. L'expansion des fronts miniers et des bassins de résidus est très visible depuis 2018.

In [ ]:
m_obi, meta_obi = make_split_map(
    site_name   = "Île Obi – Carrières de nickel (Kawasi, Indonésie)",
    lon         = 127.4302,  # 127°25'48.6"E
    lat         = -1.5479,   # 1°32'52.4"S
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 6000,
    half_h_m    = 4238,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 12
)
m_obi

### 4.7 Vietnam – Carrières de granite

> Carrières de granite entaillant le flanc d'une montagne dans le nord du Vietnam.

In [ ]:
m_vn_granite, meta_vn_granite = make_split_map(
    site_name   = "Carrières de granite (montagne, Vietnam)",
    lon         = 104.8079,  # 104°48'28.6"E
    lat         = 22.0736,   # 22°04'24.9"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 3000,
    half_h_m    = 2120,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_vn_granite

### 4.8 Makrana, Rajasthan, Inde – Carrières de marbre

> Makrana est réputée pour son marbre blanc (utilisé notamment pour le Taj Mahal). Cadrage très serré pour bien visualiser le front de taille qui recule et s'approfondit sur 8 ans.

In [ ]:
m_makrana, meta_makrana = make_split_map(
    site_name   = "Makrana – Carrière de marbre (Rajasthan, Inde)",
    lon         = 74.7089,   # 74°42'32.2"E
    lat         = 27.0271,   # 27°01'37.6"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 800,       # cadrage très serré (front de taille)
    half_h_m    = 565,      # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 16
)
m_makrana

### 4.9 Ruashi, RDC – Mines de cuivre et de cobalt

> Trois sites d'extraction proches les uns des autres, près de Lubumbashi. Emprise plus large pour couvrir l'ensemble des fosses.

In [ ]:
m_ruashi, meta_ruashi = make_split_map(
    site_name   = "Ruashi – Mines de cuivre et de cobalt (RDC)",
    lon         = 27.5703,   # 27°34'13.0"E
    lat         = -11.6240,  # 11°37'26.4"S
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 8000,
    half_h_m    = 5650,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 12
)
m_ruashi

---
## 5. 🏝️ Sites de remblais

### 5.1 Maldives – Malé, atolls à l'ouest

In [ ]:
m_maldives, meta_maldives = make_split_map(
    site_name   = "Maldives – Malé, atolls à l'ouest",
    lon         = 73.4760,   # 73°28'33.4"E
    lat         = 4.1742,    # 4°10'27.2"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 11200,
    half_h_m    = 7910,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 12
)
m_maldives

### 5.2 Genève – Baby-Plage (Lac Léman)

In [ ]:
m_geneve, meta_geneve = make_split_map(
    site_name   = "Genève – Baby-Plage (Lac Léman)",
    lon         = 6.1648,    # 6°09'52.8"E
    lat         = 46.2093,   # 46°12'33.3"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 4000,
    half_h_m    = 2830,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 14
)
m_geneve

### 5.3 Singapour – Polders côtiers (Tuas)

In [ ]:
m_singapour, meta_singapour = make_split_map(
    site_name   = "Singapour – Remblais côtiers (Tuas, côte ouest)",
    lon         = 103.7123,  # 103°42'44.6"E
    lat         = 1.2790,    # 1°16'44.4"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 11200,
    half_h_m    = 7910,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 12
)
m_singapour

### 5.4 Lagos – Eko Atlantic City, Nigeria

In [ ]:
m_lagos, meta_lagos = make_split_map(
    site_name   = "Lagos – Eko Atlantic City (Nigeria)",
    lon         = 3.4347,    # 3°26'04.8"E
    lat         = 6.4139,    # 6°24'49.9"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 5000,
    half_h_m    = 3530,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_lagos

### 5.5 Khu đô thị Cái Giá – Cát Bà, Vietnam

In [ ]:
m_catba, meta_catba = make_split_map(
    site_name   = "Cát Bà – Khu đô thị Cái Giá (Vietnam)",
    lon         = 107.0305,  # 107°01'49.9"E
    lat         = 20.7356,   # 20°44'08.0"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 5000,
    half_h_m    = 3530,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_catba

### 5.6 Hong Kong – 3ᵉ piste de l'aéroport international (Tung Chung)

> Extension par remblais au nord de l'aéroport de Hong Kong pour la construction d'une troisième piste, achevée ces dernières années.

In [ ]:
m_hk, meta_hk = make_split_map(
    site_name   = "Hong Kong – 3ᵉ piste, aéroport international (Tung Chung)",
    lon         = 113.9131,  # 113°54'47.0"E
    lat         = 22.3103,   # 22°18'37.1"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 8000,
    half_h_m    = 5650,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 12
)
m_hk

### 5.7 Manille, Philippines – Pasay Harbor City

> Remblais de sable en cours dans la baie de Manille, au niveau du port (projet toujours actif).

In [ ]:
m_manille, meta_manille = make_split_map(
    site_name   = "Manille – Pasay Harbor City (Philippines)",
    lon         = 120.9760,  # 120°58'33.6"E
    lat         = 14.5440,   # 14°32'38.5"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 5000,
    half_h_m    = 3530,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_manille

### 5.8 Abidjan, Côte d'Ivoire – Remblais dans la lagune

> La lagune Ébrié est progressivement transformée par des remblais de sable.

In [ ]:
m_abidjan, meta_abidjan = make_split_map(
    site_name   = "Abidjan – Remblais dans la lagune (Côte d'Ivoire)",
    lon         = -3.9928,   # 3°59'34.0"W
    lat         = 5.2828,    # 5°16'58.0"N
    year_before = 2018,
    year_after  = 2026,
    half_w_m    = 6000,
    half_h_m    = 4238,     # ratio A1 (1.4158) pour l'export haute résolution
    zoom        = 13
)
m_abidjan

---
## 6. 📤 Export des images (GeoTIFF 10 m + PNG A1 300 dpi)

In [ ]:
import subprocess

try:
    import rasterio
    from rasterio.plot import reshape_as_image
    import numpy as np
    from PIL import Image as PILImage
    _PNG_EXPORT_OK = True
except ImportError as e:
    print(f"⚠️  rasterio/Pillow non installé ({e}) – la conversion PNG sera ignorée")
    _PNG_EXPORT_OK = False

EXPORT_DIR = Path('./exports')
EXPORT_DIR.mkdir(exist_ok=True)

# ─── Format A1 paysage à 300 dpi ───────────────────────────────────────────────────
A1_WIDTH_PX  = 9933   # 841 mm @ 300 dpi
A1_HEIGHT_PX = 7016   # 594 mm @ 300 dpi
A1_RATIO     = A1_WIDTH_PX / A1_HEIGHT_PX  # ≈ 1.4158

# Résolution d'export fixe : 10 m (résolution native Sentinel-2 des bandes RGB)
EXPORT_SCALE_M = 10


def safe_filename(name):
    """Transforme un nom de site en nom de fichier valide."""
    return (
        name.lower()
        .replace(' ', '_')
        .replace('–', '-')
        .replace('/', '-')
        .replace('(', '').replace(')', '')
        .replace(',', '')
        .replace("'", '')
        .replace('é', 'e').replace('è', 'e').replace('ê', 'e')
        .replace('â', 'a').replace('à', 'a')
        .replace('î', 'i').replace('ô', 'o').replace('û', 'u')
    )


def resize_to_a1(img_pil, target_w=A1_WIDTH_PX, target_h=A1_HEIGHT_PX,
                  pad_color=(255, 255, 255)):
    """
    Redimensionne (agrandit si besoin, contrairement à thumbnail() qui ne
    fait que réduire) puis complète en 'letterbox' pour obtenir EXACTEMENT
    target_w x target_h px, sans déformer l'image source.

    Comme les ROI sont désormais calées sur le ratio A1 (voir sections 4/5),
    le letterbox ajouté ici est normalement minime (juste l'arrondi px).
    """
    src_w, src_h = img_pil.size
    src_ratio = src_w / src_h

    if src_ratio > target_w / target_h:
        new_w, new_h = target_w, round(target_w / src_ratio)
    else:
        new_h, new_w = target_h, round(target_h * src_ratio)

    # resize() (contrairement à thumbnail()) agrandit aussi bien qu'il réduit
    resized = img_pil.resize((new_w, new_h), PILImage.LANCZOS)

    canvas = PILImage.new('RGB', (target_w, target_h), pad_color)
    offset = ((target_w - new_w) // 2, (target_h - new_h) // 2)
    canvas.paste(resized, offset)
    return canvas


def export_site_images(meta, scale=EXPORT_SCALE_M, crs='EPSG:4326'):
    """
    Exporte les images avant/après d'un site en GeoTIFF + PNG haute résolution
    (format A1 paysage, 300 dpi).

    Paramètres
    ----------
    meta   : dict retourné par make_split_map()
    scale  : résolution en mètres pour l'export GEE (10 m = résolution
             native Sentinel-2 des bandes RGB utilisées ici — pas de
             rééchantillonnage nécessaire)
    crs    : système de projection de sortie
    """
    slug = safe_filename(meta['site_name'])
    roi  = meta['roi']

    for tag, img, yr, sat in [
        ('avant', meta['img_b'], meta['yr_b'], meta['sat_b']),
        ('apres', meta['img_a'], meta['yr_a'], meta['sat_a']),
    ]:
        tif_path = EXPORT_DIR / f"{slug}_{tag}_{yr}.tif"
        png_path = EXPORT_DIR / f"{slug}_{tag}_{yr}.png"

        print(f"  ↳ Export GeoTIFF ({scale} m) : {tif_path.name} …", end=' ', flush=True)

        # Visualiser en RGB (0–255) avant export PNG
        img_vis = img.visualize(**VIS).clip(roi)

        try:
            geemap.ee_export_image(
                img_vis,
                filename=str(tif_path),
                scale=scale,
                region=roi,
                file_per_band=False
            )
            print("✅")
        except Exception as e:
            print(f"⚠️  Erreur : {e}")
            continue

        # Conversion GeoTIFF → PNG A1 300 dpi via rasterio + Pillow
        if not _PNG_EXPORT_OK:
            continue
        print(f"  ↳ Conversion PNG A1 : {png_path.name} …", end=' ', flush=True)
        try:
            with rasterio.open(tif_path) as src:
                data = src.read()          # (bands, rows, cols)
                rgb  = reshape_as_image(data[:3])  # (rows, cols, 3)
                rgb  = np.clip(rgb, 0, 255).astype('uint8')

            img_pil = PILImage.fromarray(rgb)
            print(f"[source {img_pil.size[0]}x{img_pil.size[1]} px] ", end='', flush=True)

            img_pil = resize_to_a1(img_pil)
            img_pil.save(str(png_path), dpi=(300, 300))
            print("✅")
        except Exception as e:
            print(f"⚠️  Erreur PNG : {e}")


'''
def export_to_drive(meta, folder='Sentinel2Exports', scale=EXPORT_SCALE_M):
    """
    Alternative : export asynchrone vers Google Drive (pour très grandes zones).
    Surveiller la progression dans https://code.earthengine.google.com/tasks
    """
    slug = safe_filename(meta['site_name'])
    roi  = meta['roi']

    for tag, img, yr in [
        ('avant', meta['img_b'], meta['yr_b']),
        ('apres', meta['img_a'], meta['yr_a']),
    ]:
        task = ee.batch.Export.image.toDrive(
            image       = img.visualize(**VIS).clip(roi),
            description = f"{slug}_{tag}_{yr}",
            folder      = folder,
            fileNamePrefix = f"{slug}_{tag}_{yr}",
            region      = roi,
            scale       = scale,
            crs         = 'EPSG:4326',
            maxPixels   = 1e9
        )
        task.start()
        print(f"  ↳ Tâche Drive lancée : {slug}_{tag}_{yr} (id={task.id})")
'''

print(f"Répertoire d'export : {EXPORT_DIR.resolve()}")
print(f"Format cible PNG : A1 paysage, {A1_WIDTH_PX}x{A1_HEIGHT_PX} px (300 dpi)")
print(f"Résolution d'export GeoTIFF : {EXPORT_SCALE_M} m (natif Sentinel-2)")

In [ ]:
# ── Export de tous les sites ───────────────────────────────────────────────────
# Export local systématique : GeoTIFF @ scale=10 m + PNG A1 300 dpi.
# Remplacer export_site_images par export_to_drive pour passer par Google Drive.

all_metas = [
    # Sites d'extraction
    meta_carrare,
    meta_saleve,
    meta_arana,
    meta_msh,
    meta_siam,
    meta_obi,
    meta_vn_granite,
    meta_makrana,
    meta_ruashi,
    # Sites de remblais
    meta_maldives,
    meta_geneve,
    meta_singapour,
    meta_lagos,
    meta_catba,
    meta_hk,
    meta_manille,
    meta_abidjan,
]

for meta in all_metas:
    print(f"\n📦  {meta['site_name']}")
    export_site_images(meta, scale=EXPORT_SCALE_M)

print("\n✅  Export terminé. Fichiers dans :", EXPORT_DIR.resolve())

---
## 7. 🎞️ GIF Timelapse (2017 → présent)

Pour chaque site, un GIF animé est généré en utilisant `geemap.sentinel2_timelapse()`, qui s'appuie sur les collections Sentinel-2 annuelles disponibles dans GEE.

Les GIF sont sauvegardés dans `./exports/timelapse/`.

**Paramètres clés :**
- `bands` : RGB naturel (Red/Green/Blue)
- `frequency` : fréquence des images ('year')
- `dimensions` : largeur en pixels (hauteur calculée automatiquement)
- `frames_per_second` : vitesse d'animation
- `title` : texte affiché sur le GIF (nom du site)

In [ ]:
from IPython.display import Image as IPyImage, display

TIMELAPSE_DIR = EXPORT_DIR / 'timelapse'
TIMELAPSE_DIR.mkdir(exist_ok=True)


def make_timelapse(meta, fps=3, width_px=600):
    """
    Génère un GIF timelapse Sentinel-2 pour un site donné.

    Paramètres
    ----------
    meta      : dict retourné par make_split_map()
    fps       : images par seconde dans le GIF
    width_px  : largeur du GIF en pixels (hauteur proportionnelle)
    """
    slug     = safe_filename(meta['site_name'])
    gif_path = str(TIMELAPSE_DIR / f"{slug}_timelapse.gif")
    roi      = meta['roi']

    # Calculer les années de début/fin (bornées à la disponibilité Sentinel-2)
    start_year = max(meta['yr_b'], S2_FIRST_YEAR)
    end_year   = meta['yr_a']

    # Ratio largeur/hauteur à partir de l'emprise
    ratio  = meta['half_w_m'] / meta['half_h_m']
    height_px = int(width_px / ratio)

    print(f"  🎞️  Timelapse {start_year}–{end_year} ({width_px}×{height_px} px) …",
          end=' ', flush=True)

    try:
        geemap.sentinel2_timelapse(
            roi,
            out_gif           = gif_path,
            start_year        = start_year,
            end_year           = end_year,
            start_date         = '04-01',
            end_date           = '10-31',
            frequency          = 'year',
            bands              = ['Red', 'Green', 'Blue'],   # RGB naturel
            frames_per_second  = fps,
            title              = meta['site_name'],
            dimensions         = width_px,
        )
        print("✅")
        return gif_path
    except Exception as e:
        print(f"⚠️  Erreur : {e}")
        return None


print(f"Répertoire timelapse : {TIMELAPSE_DIR.resolve()}")

In [ ]:
# ── Génération de tous les timelapses ─────────────────────────────────────────
# Note : chaque appel peut prendre 1–3 minutes selon la taille de la zone.

gif_paths = {}

for meta in all_metas:
    print(f"\n🎬  {meta['site_name']}")
    gif = make_timelapse(meta, fps=3, width_px=600)
    if gif:
        gif_paths[meta['site_name']] = gif

print(f"\n✅  {len(gif_paths)}/{len(all_metas)} timelapse(s) générés.")

In [ ]:
# ── Affichage inline des GIF dans le notebook ──────────────────────────────────

for site_name, gif_path in gif_paths.items():
    print(f"\n🎞️  {site_name}")
    display(IPyImage(filename=gif_path, width=600))

---
## 8. Notes méthodologiques

| Paramètre | Valeur |
|-----------|--------|
| **Satellite** | Sentinel-2 MSI (couverture globale fiable à partir de 2017) |
| **Collection** | `COPERNICUS/S2_SR_HARMONIZED` (niveau 2A, réflectance de surface, harmonisée) |
| **Résolution** | 10 m (natif, bandes B4/B3/B2) |
| **Fenêtre temporelle** | Avril – Octobre (minimiser les nuages et la neige) |
| **Masque nuages** | Bande `SCL` (ombres, nuages moyenne/haute proba., cirrus fins) |
| **Filtrage nuages** | `CLOUDY_PIXEL_PERCENTAGE` < 20 % (fallback 40 % sur l'année complète) |
| **Composite** | Médiane de la collection filtrée |
| **Visualisation** | Couleurs naturelles (RGB) ; min=0.0, max=0.3, gamma=1.4 |
| **Année "avant"** | Cible 2018 ; si peu d'images exploitables, recherche l'année la plus proche (± 4 ans) |
| **Année "après"** | Cible 2026 ; même logique de recherche autour de la cible |
| **ROI** | Rectangle centré sur le site ; dimensions calées sur le ratio A1 |
| **Export** | GeoTIFF 10 m + PNG 300 dpi (A1 paysage, systématique) via `rasterio` + `Pillow` |
| **Timelapse** | `geemap.sentinel2_timelapse()` ; GIF animé 600 px, 3 fps |

### Limites
- La recherche d'année "autour" de la cible peut faire varier légèrement l'année effective d'un site à l'autre selon la disponibilité d'images propres.
- Certains sites (Mont-Saint-Hilaire) évoluent principalement en profondeur : l'empreinte surfacique peut sembler stable.
- Les timelapses sont sensibles à la disponibilité des images (certaines années peuvent être sautées si nuageuses).

### Pour aller plus loin
- Calculer des indices spectraux (NDVI, NDWI, MNDWI) pour quantifier les changements de végétation ou de surfaces en eau.
- Exporter en **COG** (Cloud Optimized GeoTIFF) pour partage et publication web.
- Affiner le masque de nuages avec `s2cloudless` pour les sites les plus fréquemment nuageux.